# Verify host-expression source fidelity gaps and pending version selection


Recompute every reported installed-source replay count and preserve discrepancies as gaps. Verify source-provider and acquisition hashes, full selected gene rows and exact primary Atlas table equivalence. Run the selection policy with explicit pending admission; a newer release or more annotation rows cannot force promotion before quality and mapping review.

In [1]:
import json, hashlib, gzip, base64
import pandas as pd, numpy as np
from pathlib import Path
from starplast import dataset_selection as DS
from starplast import host as H, organisms as O
base=Path('results/host_expression_source_review_2026_10_08')
out=base/'comparison_v3'
reports=json.loads((out/'comparison.json').read_text())
records={r['source_id']:r for r in json.loads((base/'sources.json').read_text())}
for version,key in [('v10','host_gtex_transcriptome'),('v11','host_gtex_transcriptome')]:
    binding=records[key]['file'] if version=='v10' else records[key]['candidate_v11']['file']
    data=Path(binding['path']).read_bytes()
    assert hashlib.sha256(data).hexdigest()==binding['sha256']
    if version=='v11':
        expected=records[key]['candidate_v11']['object_metadata']
        assert base64.b64encode(hashlib.md5(data).digest()).decode()==expected['md5Hash']
    with gzip.open(binding['path'],'rt') as stream:
        stream.readline(); stream.readline()
        raw=pd.read_csv(stream,sep='\t')
    selected=pd.read_parquet(out/(version+'_selected_gene_rows.parquet'))
    assert selected.Name.tolist()==raw.Name.tolist() and selected.ensembl_gene.is_unique
    for column in ('Name','Description',*H.GTEX_TISSUES):
        pd.testing.assert_series_equal(selected[column],raw[column])
    print(version,'all selected original gene records retained',len(selected),'native core duplicate rows',int(selected.native_core_gene.duplicated(keep=False).sum()))
for key,record in reports['installed_reproduction'].items():
    for column,metrics in record['fields'].items():
        rows=pd.read_parquet(out/(key+'_'+column+'_replay_rows.parquet'))
        assert rows.host_id.is_unique
        measured=rows.installed_value.notna()
        replayed=rows.replayed_value.notna()
        assert int(measured.sum())==metrics['installed_observed']
        assert int((measured & ~replayed).sum())==metrics['missing_reproductions']
        assert int((~measured & replayed).sum())==metrics['extra_mapped_values_on_installed_universe']
        overlap=measured & replayed
        assert int(rows.loc[overlap,'installed_value'].ne(rows.loc[overlap,'replayed_value']).sum())==metrics['exact_differences']
        print(key,column,'replay counts verified; differences',metrics['exact_differences'],'missing',metrics['missing_reproductions'],'extra',metrics['extra_mapped_values_on_installed_universe'])

v10 all selected original gene records retained 59033 native core duplicate rows 90
v11 all selected original gene records retained 74628 native core duplicate rows 0
v10 fibroblast_tpm replay counts verified; differences 0 missing 3 extra 2
v10 hepatocyte_tpm replay counts verified; differences 0 missing 3 extra 2
fantom brain_tpm replay counts verified; differences 1741 missing 8 extra 1342
fantom skeletal_muscle_tpm replay counts verified; differences 0 missing 7 extra 1276


In [2]:
receipt=json.loads((out/'fantom_primary_equivalence.json').read_text())
original=Path('/media/carruthers/mnt3/claude/toxoplasma_projects/datasets/host_expression_review_inputs/FANTOM5_E-MTAB-3579/tpmss.tsv')
historical=Path(next(f['absolute_path'] for f in records['host_mouse_tissue_transcriptome']['verified_files'] if f['name']=='E-MTAB-3579-mouse-tissue-tpm.tsv'))
assert hashlib.sha256(original.read_bytes()).hexdigest()==receipt['sha256']
assert hashlib.sha256(historical.read_bytes()).hexdigest()==receipt['historical_sha256']
pd.testing.assert_frame_equal(pd.read_csv(original,sep='\t',comment='#'),pd.read_csv(historical,sep='\t',comment='#'))
lcm=json.loads((base/'lcm_readme_receipt.json').read_text())
assert lcm['status']=='primary_object_MD5_size_verified'
assert hashlib.sha256(Path(lcm['path']).read_bytes()).hexdigest()==lcm['sha256']
candidates=[]
publication=json.loads((base/'host_gtex_transcriptome_publication.json').read_text())['resultList']['result'][0]
for version in ('v10','v11'):
    candidates.append(DS.DatasetCandidate(dataset_id='GTEx_'+version,scope='human selected fibroblast/hepatocyte reference RNA-seq TPM',publication_id='PMID:'+publication['id'],first_publication_date=publication['firstPublicationDate'],citations=publication.get('citedByCount'),citation_provider='Europe PMC',citation_snapshot='2026-10-08',admission='pending',admission_reason='release-to-publication lineage, native mapping drift/PAR_Y, fixed eligible coverage and affected validation pending'))
evaluated=DS.rank_candidates(candidates,'2026-10-08')
assert all(r['status']=='pending' and r['score'] is None for r in evaluated)
assert DS.preferred(candidates,'2026-10-08') is None
(base/'selection_review.json').write_text(json.dumps({'policy':DS.SelectionPolicy().to_dict(),'candidates':evaluated,'decision':'retain_installed_values_pending_mapping_transform_review; candidate_not_promoted','recency_interpretation':'v11 release does not reset the shared publication citation-age clock','coverage_interpretation':'gene-annotation rows are not measured protein comprehensiveness'},indent=2,allow_nan=False)+'\n')
print('Atlas primary table exact, LCM receipt exact, selection policy correctly refuses premature promotion.')

Atlas primary table exact, LCM receipt exact, selection policy correctly refuses premature promotion.


Public LCM README identifies laser microdissection of fixed tissue, 172 sequenced samples from five tissue types including targeted liver hepatocytes and portal tract. Cultured fibroblast medians are a reference context, not proof of matching infection-study donor/culture. Native mapping replay is not exact for installed host sources; neither v11 nor the rederived Atlas values is promoted. Continue the mapping/transform review and other independent tasks.